# Baseline 2 — image-only CNN for glaucoma prescreening

**Team:** Indira Duisembayeva, Aman Khan, Yomna Elnikety  
**Prepared:** 4 October 2026  
**Default:** ImageNet-pretrained ResNet-18, full 224 × 224 RGB fundus photographs.

This notebook explains the supplied preprocessing handoff, audits its frozen manifests,
then trains and evaluates a binary CNN. All implementation functions are embedded here;
you do not need to import `baseline_core.py` or the teammate's loader to run it.

**Target meaning matters:** in CHAKSU, 1 means **expert-labelled glaucoma suspect** and
0 means normal. In PAPILA, 1 means clinical glaucoma and 0 means healthy; the separate
suspect class (2) is excluded. The model outputs an **uncalibrated sigmoid suspicion score**
between 0 and 1. Calling it a clinically validated probability of glaucoma would overstate
the evidence. The external experiment tests transfer across both acquisition and label definitions.

**Current status:** the supplied folder has code, exploration notebooks and evidence summaries.
It does **not** contain prepared photographs, `prepared_manifest.csv`, or actual split CSVs.
No CNN training results are supplied, and no real-data performance is claimed by this notebook.
Obtain the completed `artifacts/` export from your teammate before the training cells can run.
The missing-data check explains exactly what is needed.

## 1. What your teammate did and what the evidence shows

She implemented `inventory → prepare → split`, with two exploratory notebooks and 11
synthetic unit tests. This is **data preparation evidence**, not model performance.

1. **Inventory and label matching.** Matched 1,345 CHAKSU photographs (three cameras,
   official train/test releases) and 488 PAPILA photographs (244 patients, both eyes).
   CHAKSU labels were cross-checked against the majority of five expert votes. The code
   accepts the released `Majority Decision` / `Glaucoma Decision` column variations and
   a label spelling error. Bosch tables list all 145 images in both releases, so the join
   retains only rows whose image exists in that release.
2. **Image preparation.** Creates 224 × 224 classification images by centring the full
   RGB photograph in a black square and resizing with bilinear interpolation. There is
   no disc crop. Separately prepares 512 × 512 segmentation inputs and masks. It checks
   EXIF orientation and stops on an unhandled nontrivial orientation. The handoff loader
   applies fixed ImageNet RGB normalization at loading time; PNGs are not normalized twice.
3. **Anatomical reference preparation.** Uses CHAKSU STAPLE masks and PAPILA expert-1
   contours. Computes expert CDR/area measurements into a separate oracle-only file.
   These measurements, masks, votes and clinical variables are not inputs to this baseline.
4. **Geometry checks.** Flags 33 of 1,833 reference masks for cup pixels outside the disc.
   The supplied review records all 33 as boundary artefacts; none has an empty cup/disc.
   They remain available for classification but are excluded from segmentation eligibility.
5. **Duplicate review.** Records seven exact-pixel duplicate pairs (six Bosch train,
   one Bosch test). The revised review CSV contains 26 candidate near-duplicate pairs,
   all recorded as `not_duplicate`. Identity and duplicate components are kept together
   by the original splitter; no release/external overlap was quarantined in that run.
6. **Evaluation roles.** Divides CHAKSU's training release into development train and
   validation, reserves CHAKSU's test release, and keeps PAPILA external. PAPILA's 68
   suspect images are excluded from binary classification, leaving 420.

These are the **original handoff counts**, not guaranteed counts after the precautionary
identity correction below:

| Partition | Negative | Positive | Total | Positive definition |
|---|---:|---:|---:|---|
| CHAKSU train | 698 | 109 | 807 | glaucoma suspect |
| CHAKSU validation | 174 | 28 | 202 | glaucoma suspect |
| CHAKSU held-out test | 285 | 51 | 336 | glaucoma suspect |
| PAPILA external test | 333 | 87 | 420 | clinical glaucoma |

CHAKSU train positives are 13.5%. Always-negative prediction would achieve **86.5% training
accuracy**, demonstrating why accuracy is not a sufficient success metric. No majority-class
accuracy is presented as a trained CNN result.

The original segmentation counts are 787 / 200 / 327 / 486. The PAPILA segmentation count
can exceed its classification count because an ambiguous diagnosis does not invalidate
its anatomical contours. Classification must use `classification_eligible`, not segmentation eligibility.

## 2. Issues found in the handoff audit

**Potential patient leakage overlooked by the mapper.** `quality_review.csv` includes 19
Bosch **training** images with `P1`–`P10` names, although the prose and mapping script say
that only Bosch test filenames have this pattern. It contains `chaksu_Bosch_train_P10_Image1`;
`patient_groups.csv` contains `chaksu_Bosch_test_P10_Image2`. This is a candidate shared
identity across official releases, not proof of identical photographs or confirmed patient identity.
The supplied training/validation CSVs are absent, so we cannot establish which training
pairs were split apart. The original patient mapping contains 41 test images, representing
21 filename IDs, not 41 unique patients.

**Policy used here, declared before training:** conservatively join the same Bosch `P` number
across both releases, along with all existing patient/duplicate links. Quarantine any whole
component that spans train/test releases. If a newly linked component spans development
train and validation, move its complete component to train. This prevents the identified
possible overlap without asserting that filename IDs are verified patients. Save the derivative
manifest and every change separately; leave the original files untouched. At least the two
P10 images are candidates for quarantine if present in the actual supplied manifests; transitive
links may add more. The effective counts are computed from actual manifests below.

**Remaining identity limitation.** The original split is `PROVISIONAL`: it lacks verified
patient identity for 1,304 CHAKSU images. Filename grouping does not establish full patient
disjointness or resolve repeat visits under other names. Report internal results and confidence
intervals as provisional; do not invent patient IDs. PAPILA has genuine patient IDs, allowing
clustered bootstrap intervals for its eye-level metrics.

**Review coverage and stale summaries.** The quality CSV records 33 `confirmed_artifact`,
12 `confirmed_ok`, and 1,788 `pending` rows. The 45 reviewed rows are flagged/sampled
mask-overlay reviews, not a complete image-gradability assessment. `preparation_summary.json`
still reports the initial 50,333 near-duplicate candidates and pending visual review; the later
review CSV reports 26 reviewed candidates. Both describe different stages. The main `prepare`
function still uses the old detector; the revised detector is a separate script whose default
threshold is 8, while the documented run used 11. A fresh pipeline run therefore requires the
missing intermediate step, not just the README commands.

Same-camera hashing reduces false positives but cannot rule out cross-camera duplicates or
the same eye photographed on another device. Also, all three CHAKSU cameras occur in the
original development/test tiers: this baseline is **not a held-out-camera domain-generalization
experiment**. PAPILA measures external transfer, with an additional label-definition shift.

**Reproduction details:** executables live in `script/` (and `script/dev/`), although some README
commands omit those prefixes. The exploration notebooks have no saved executed outputs;
their written findings are corroborated where possible by the JSON/CSV evidence. No claim
of zero leakage or completed model training can be made from this handoff.

## 3. Environment and files to obtain

Use Python **3.11–3.13**. The implementation was checked locally with Python 3.13,
PyTorch 2.9.1 and torchvision 0.24.1. Start Jupyter in an environment containing the packages
below; package installation is optional and commented out. Restart the kernel after installation.
For a CUDA installation, use the matching PyTorch/torchvision packages from the official
[PyTorch installer](https://pytorch.org/get-started/locally/).

Request this exact completed export from your teammate:

```text
artifacts/
  prepared_manifest.csv
  splits/
    all.csv
    split_summary.json
    train_classification.csv
    validation_classification.csv
    in_domain_test_classification.csv
    external_test_classification.csv
  prepared/
    chaksu/<image_id>/image_224.png
    papila/<image_id>/image_224.png
```

Keep the relative image paths intact. The evidence-summary folder is not a substitute for
`artifacts/`. Checksum validation deliberately rejects altered frozen manifests. The notebook
reads held-out metadata for leakage auditing but does not open held-out photographs during
development. No model or optimizer is fitted on PAPILA.

Default ResNet-18 ImageNet weights are about 45 MB and download on first use if not cached.
A failed download raises an error; the notebook does not silently fall back to random weights.

In [ ]:
# Optional: uncomment in a fresh Python 3.11–3.13 kernel, run, then restart the kernel.
# %pip install torch==2.9.1 torchvision==0.24.1 numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1 Pillow==11.1.0 matplotlib==3.10.0
import sys
print("Notebook interpreter:", sys.executable)
print("Python:", sys.version)


### Imports and file-integrity helpers

JSON outputs represent undefined statistics as null.

In [ ]:
"""Image-only baseline helpers. Embedded verbatim in the delivered notebook.

Only RGB photographs are model inputs. Labels/identities serve training/auditing.
No expert masks, measurements, votes, camera IDs, or clinical covariates enter CNN.
"""
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import random
import re
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, confusion_matrix,
    brier_score_loss, log_loss, precision_recall_curve,
)

PARTITIONS = ('train', 'validation', 'in_domain_test', 'external_test')
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, value):
    # Invalid/undefined metrics are represented by null, never nonstandard NaN.
    def clean(obj):
        if isinstance(obj, dict):
            return {str(k): clean(v) for k, v in obj.items()}
        if isinstance(obj, (tuple, list)):
            return [clean(v) for v in obj]
        if isinstance(obj, np.generic):
            return clean(obj.item())
        if isinstance(obj, float) and not np.isfinite(obj):
            return None
        if isinstance(obj, Path):
            return str(obj)
        return obj
    Path(path).write_text(json.dumps(clean(value), indent=2, allow_nan=False) + '\n')


def read_table(path):
    # Empty patient IDs must stay empty, rather than becoming the string 'nan'.
    return pd.read_csv(path, dtype=str, keep_default_na=False)


### Frozen-manifest audit and precautionary grouping

Known identities and duplicate hashes cannot cross active tiers. Filename grouping is precautionary.

In [ ]:
class Components:
    def __init__(self, ids):
        self.parent = {x: x for x in ids}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def join(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.parent[max(a, b)] = min(a, b)


def bosch_filename_key(row):
    # Precautionary identity constraint, NOT a claim of verified patient identity.
    if row['dataset'] == 'chaksu' and row['camera'] == 'Bosch':
        match = re.fullmatch(r'chaksu_Bosch_(?:train|test)_P(\d+)_Image[12]', row['image_id'], re.I)
        if match:
            return 'possible_bosch_P' + str(int(match.group(1)))
    return ''


def load_and_audit(artifacts):
    """Validate the real frozen manifest before selecting any training rows.

    Reads test metadata only to audit identity leakage; does not open test images.
    Counts in each CSV must agree with the hashed all.csv, not merely a summary.
    """
    root = Path(artifacts).expanduser().resolve()
    required = [root / 'prepared_manifest.csv', root / 'splits/all.csv',
                root / 'splits/split_summary.json']
    required += [root / 'splits' / f'{p}_classification.csv' for p in PARTITIONS]
    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            'The handoff summaries are not training data. Obtain the completed artifacts/ '
            'folder from your teammate (prepared images, prepared_manifest.csv, splits/). '
            'Do not invent/recreate the frozen split from the reported counts. Missing:\n' + '\n'.join(missing))
    summary = json.loads((root / 'splits/split_summary.json').read_text())
    for key, path in [('all_splits_sha256', root / 'splits/all.csv'),
                      ('manifest_sha256', root / 'prepared_manifest.csv')]:
        if not summary.get(key) or sha256_file(path) != summary[key]:
            raise ValueError(f'Frozen data integrity check failed: {key}')
    all_rows = read_table(root / 'splits/all.csv')
    fields = {'image_id', 'dataset', 'chaksu_release', 'camera', 'patient_id',
              'patient_id_verified', 'pixel_sha256', 'group_id', 'split',
              'classification_eligible', 'binary_label', 'original_label', 'classification_image'}
    if fields - set(all_rows):
        raise ValueError(f'Missing manifest columns: {sorted(fields - set(all_rows))}')
    if all_rows.empty or all_rows.image_id.duplicated().any() or (all_rows.image_id == '').any():
        raise ValueError('Empty or repeated image identity in all.csv')
    if not all_rows.classification_eligible.isin(['0', '1']).all():
        raise ValueError('Invalid classification eligibility flag')
    if (all_rows.pixel_sha256 == '').any() or (all_rows.group_id == '').any():
        raise ValueError('Missing duplicate hash or frozen group identity')
    for partition in PARTITIONS:
        table = read_table(root / 'splits' / f'{partition}_classification.csv')
        expected = all_rows[(all_rows.split == partition) & (all_rows.classification_eligible == '1')]
        if set(table.columns) != set(all_rows.columns):
            raise ValueError(f'{partition}: CSV columns differ from all.csv')
        left = table.sort_values('image_id').reset_index(drop=True)[all_rows.columns]
        right = expected.sort_values('image_id').reset_index(drop=True)
        if not left.equals(right):
            raise ValueError(f'{partition}: rows differ from frozen all.csv')
        if len(table) != summary.get('classification_counts', {}).get(partition):
            raise ValueError(f'{partition}: count differs from summary')
    active = all_rows[all_rows.split.isin(PARTITIONS)]
    for _, r in active.iterrows():
        expected = ('papila', '') if r['split'] == 'external_test' else (
            'chaksu', 'test' if r['split'] == 'in_domain_test' else 'train')
        if (r['dataset'], r['chaksu_release']) != expected:
            raise ValueError(f'Dataset/release crossed evaluation roles: {r.image_id}')
        if r.classification_eligible == '1':
            if r.binary_label not in ('0', '1'):
                raise ValueError(f'Nonbinary classification target: {r.image_id}')
            original = ' '.join(r.original_label.upper().split()).replace('SUSUPECT', 'SUSPECT')
            mapping = {'NORMAL': '0', 'GLAUCOMA SUSPECT': '1'} if r.dataset == 'chaksu' else {'0': '0', '1': '1'}
            if mapping.get(original) != r.binary_label:
                raise ValueError(f'Unexpected label mapping: {r.image_id}')
        if r.dataset == 'papila' and (not r.patient_id or r.patient_id_verified != '1'):
            raise ValueError('PAPILA requires verified patient identities')
    if summary.get('near_duplicate_reviews_pending', 0):
        raise ValueError('Resolve pending duplicate review before training')
    # Existing known identities must already be disjoint across active tiers.
    for field in ('patient_id', 'pixel_sha256', 'group_id'):
        subset = active[active[field] != '']
        if (subset.groupby(field)['split'].nunique() > 1).any():
            raise ValueError(f'Known {field} crosses splits; repair upstream before training')
    return root, all_rows, summary


def effective_partitions(all_rows):
    """Create a conservative derivative split without altering the original.

    Merge possible P-number identities with existing transitive duplicate groups.
    Quarantine entire components spanning source releases. If a possible identity
    spans train/validation, put its full development component into TRAIN.
    The deterministic policy uses no pixel interpretation or model performance.
    """
    df = all_rows.copy()
    df['original_split'] = df['split']
    df['possible_filename_group'] = [bosch_filename_key(r) for _, r in df.iterrows()]
    uf = Components(df.image_id.tolist())
    for field in ('group_id', 'patient_id', 'pixel_sha256', 'possible_filename_group'):
        first = {}
        for _, r in df.iterrows():
            key = r[field]
            if key:
                if key in first:
                    uf.join(first[key], r.image_id)
                else:
                    first[key] = r.image_id
    df['analysis_group'] = [uf.find(x) for x in df.image_id]
    df['split_change_reason'] = ''
    for _, indices in df.groupby('analysis_group').groups.items():
        group = df.loc[indices]
        datasets = set(group.dataset)
        releases = set(group.loc[group.dataset == 'chaksu', 'chaksu_release'])
        old = set(group.split)
        reason = None
        if any(s.startswith('quarantine') for s in old):
            destination, reason = 'quarantine_inherited', 'component includes previously quarantined image'
        elif len(datasets) > 1:
            destination, reason = 'quarantine_cross_dataset', 'possible shared identity across datasets'
        elif len(releases) > 1:
            destination, reason = 'quarantine_possible_patient_overlap', 'possible Bosch filename identity spans releases'
        elif old == {'train', 'validation'}:
            destination, reason = 'train', 'possible filename identity crosses development split; keep together in train'
        if reason:
            df.loc[indices, 'split'] = destination
            df.loc[indices, 'split_change_reason'] = reason
    active = df[df.split.isin(PARTITIONS)]
    if (active.groupby('analysis_group')['split'].nunique() > 1).any():
        raise ValueError('Effective groups still cross evaluation tiers')
    partitions = {}
    for p in PARTITIONS:
        rows = df[(df.split == p) & (df.classification_eligible == '1')].copy().reset_index(drop=True)
        if set(rows.binary_label) != {'0', '1'}:
            raise ValueError(f'{p}: precautionary grouping leaves fewer than two classes; revise protocol')
        rows['binary_label'] = rows.binary_label.astype(int)
        partitions[p] = rows
    return df, partitions


### RGB image loading, normalization and reproducibility

Only the classification photograph is passed to the model. DataLoader uses zero workers for portable Jupyter execution.

In [ ]:
def image_path(root, relative):
    relative = Path(relative)
    if relative.is_absolute():
        raise ValueError('classification_image must be relative to artifacts/')
    path = (Path(root) / relative).resolve()
    if not path.is_relative_to(Path(root).resolve()):
        raise ValueError('Image path escapes artifacts/')
    if path.name != 'image_224.png' and path.name != 'image_classification.png':
        raise ValueError('Expected the pipeline RGB classification image, not a mask or overlay')
    return path


def check_images(root, rows):
    # Call on development only until final evaluation is explicitly enabled.
    for r in rows.itertuples():
        path = image_path(root, r.classification_image)
        if not path.is_file():
            raise FileNotFoundError(f'Missing prepared image: {path}')
        with Image.open(path) as im:
            if im.mode != 'RGB' or im.size != (224, 224):
                raise ValueError(f'Expected original pipeline RGB 224x224 export: {path}')
            im.load()


def make_transform(training=False, augment=True):
    operations = []
    if training and augment:
        # Full image retained; no random crops or anatomical orientation changes.
        operations.append(transforms.ColorJitter(brightness=.10, contrast=.10, saturation=.05))
    return transforms.Compose(operations + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class FundusDataset(Dataset):
    def __init__(self, rows, root, training=False, augment=True):
        self.rows = rows.reset_index(drop=True).copy()
        self.root = Path(root)
        self.transform = make_transform(training, augment)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        with Image.open(image_path(self.root, row.classification_image)) as image:
            x = self.transform(image.convert('RGB'))
        return x, torch.tensor(float(row.binary_label), dtype=torch.float32), row.image_id


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    # Warn if the chosen accelerator lacks deterministic support; record device.
    torch.use_deterministic_algorithms(True, warn_only=True)


def pick_device(request='auto'):
    if request != 'auto':
        return torch.device(request)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def make_loader(rows, root, config, training=False):
    # workers=0 works when Dataset is defined in a Windows/macOS Jupyter cell.
    generator = torch.Generator().manual_seed(config['seed'])
    return DataLoader(FundusDataset(rows, root, training, config['augment']),
                      batch_size=config['batch_size'], shuffle=training,
                      num_workers=0, drop_last=False, generator=generator)


### Model initialization and training

ResNet-18 is the default. Train the binary head for two epochs, then fine-tune the whole encoder. Keep BatchNorm running statistics fixed; choose a fine-tuned checkpoint by validation AUROC, breaking ties with validation BCE.

In [ ]:
def make_model(name, pretrained=True):
    builders = {
        'resnet18': (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1),
        'resnet50': (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V2),
        'efficientnet_b0': (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
    }
    if name not in builders:
        raise ValueError(f'Unknown architecture: {name}')
    builder, weights = builders[name]
    # Weight download failures must propagate: never silently train from scratch.
    model = builder(weights=weights if pretrained else None)
    if name.startswith('resnet'):
        model.fc = nn.Linear(model.fc.in_features, 1)
    else:
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    return model, str(weights) if pretrained else 'NONE_TEST_ONLY'


def head_parameters(model, name):
    return list(model.fc.parameters()) if name.startswith('resnet') else list(model.classifier.parameters())


def training_mode(model, head_only=False):
    if head_only:
        # Frozen backbone: freeze BatchNorm buffers and EfficientNet dropout too.
        model.eval()
    else:
        model.train()
    # Keep pretrained normalization statistics stable for small/imbalanced data.
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


def run_epoch(model, loader, device, optimizer=None, head_only=False):
    training = optimizer is not None
    if training:
        training_mode(model, head_only)
    else:
        model.eval()
    losses, targets, probs, ids = 0., [], [], []
    # Unweighted BCE estimates the original training-distribution posterior.
    # Class imbalance is addressed by ranking metrics and a validation threshold.
    criterion = nn.BCEWithLogitsLoss()
    with torch.set_grad_enabled(training):
        for x, y, image_ids in loader:
            x, y = x.to(device), y.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(x).reshape(-1)
            if logits.shape != y.shape:
                raise ValueError('Logit/target shape mismatch')
            loss = criterion(logits, y)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite loss')
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.)
                optimizer.step()
            losses += loss.item() * len(y)
            targets.extend(y.detach().cpu().numpy().astype(int).tolist())
            probs.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
            ids.extend(image_ids)
    if not targets:
        raise ValueError('Empty data loader')
    pred = pd.DataFrame({'image_id': ids, 'label': targets, 'probability': probs})
    return losses / len(targets), pred


def fit_model(train_rows, val_rows, root, out, config, device, pretrained=True):
    out = Path(out)
    if (out / 'best.pt').exists():
        raise FileExistsError('A checkpoint exists. Use a new run directory for a new experiment.')
    if config['epochs'] <= config['warmup_epochs']:
        raise ValueError('epochs must exceed warmup_epochs so the backbone is fine-tuned')
    seed_everything(config['seed'])
    model, weight_id = make_model(config['model_name'], pretrained)
    model.to(device)
    head = head_parameters(model, config['model_name'])
    head_ids = {id(p) for p in head}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    train_loader = make_loader(train_rows, root, config, training=True)
    val_loader = make_loader(val_rows, root, config)
    best_auc, best_loss, bad_epochs, history = -np.inf, np.inf, 0, []
    optimizer = None
    for epoch in range(config['epochs']):
        head_only = epoch < config['warmup_epochs']
        if epoch == 0 or epoch == config['warmup_epochs']:
            for p in backbone:
                p.requires_grad_(not head_only)
            groups = [{'params': head, 'lr': config['head_lr']}]
            if not head_only:
                groups.append({'params': backbone, 'lr': config['backbone_lr']})
            optimizer = torch.optim.AdamW(groups, weight_decay=config['weight_decay'])
            bad_epochs = 0
        started = time.monotonic()
        train_loss, _ = run_epoch(model, train_loader, device, optimizer, head_only)
        val_loss, predictions = run_epoch(model, val_loader, device)
        auc = float(roc_auc_score(predictions.label, predictions.probability))
        ap = float(average_precision_score(predictions.label, predictions.probability))
        entry = dict(epoch=epoch + 1, phase='head' if head_only else 'fine_tune',
                     train_bce=train_loss, validation_bce=val_loss,
                     validation_auroc=auc, validation_average_precision=ap,
                     seconds=time.monotonic() - started)
        history.append(entry)
        pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
        # Compare only fine-tuning checkpoints: selected model is end-to-end tuned.
        improved = not head_only and (auc > best_auc + 1e-8 or (abs(auc - best_auc) <= 1e-8 and val_loss < best_loss))
        if improved:
            best_auc, best_loss, bad_epochs = auc, val_loss, 0
            checkpoint = {'model_state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                          'config': dict(config), 'weights': weight_id, 'best_epoch': epoch + 1,
                          'validation_auroc': auc, 'validation_bce': val_loss}
            torch.save(checkpoint, out / 'best.pt')
        elif not head_only:
            bad_epochs += 1
        print(f"Epoch {epoch + 1:02d} [{entry['phase']}]: train BCE={train_loss:.4f}; "
              f"val BCE={val_loss:.4f}, AUROC={auc:.4f}, AP={ap:.4f}", flush=True)
        if not head_only and bad_epochs >= config['patience']:
            print('Early stopping on validation AUROC (BCE breaks ties).')
            break
    model, checkpoint = restore_model(out / 'best.pt', device)
    return model, checkpoint, pd.DataFrame(history)


def restore_model(path, device):
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    model, _ = make_model(checkpoint['config']['model_name'], pretrained=False)
    model.load_state_dict(checkpoint['model_state'], strict=True)
    model.to(device).eval()
    return model, checkpoint


### Metrics, threshold selection and clustered uncertainty

Average precision (AP) is the reported precision–recall summary; it is not trapezoidal PR area. Choose one threshold on validation to maximize specificity subject to at least 90% validation sensitivity. Test sensitivity is subsequently measured, never enforced by reselecting the threshold.

In [ ]:
def checked_arrays(y, p):
    y, p = np.asarray(y), np.asarray(p, dtype=float)
    if y.ndim != 1 or p.shape != y.shape or len(y) == 0:
        raise ValueError('Expected nonempty 1D labels and probabilities of equal length')
    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Expected binary labels and finite probabilities in [0,1]')
    return y.astype(int), p


def select_threshold(y, p, target_sensitivity=.90):
    y, p = checked_arrays(y, p)
    if len(np.unique(y)) != 2 or not 0 < target_sensitivity <= 1:
        raise ValueError('Threshold selection needs both classes and target sensitivity in (0,1]')
    fpr, tpr, thresholds = roc_curve(y, p, drop_intermediate=False)
    good = np.flatnonzero(np.isfinite(thresholds) & (tpr >= target_sensitivity))
    # Maximize specificity subject to required validation sensitivity. Ties use
    # the largest threshold. This does not promise the same sensitivity on test.
    chosen = max(good, key=lambda i: (1 - fpr[i], thresholds[i]))
    return float(thresholds[chosen])


def metric_values(y, p, threshold=.5):
    y, p = checked_arrays(y, p)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError('Threshold must be finite and in [0,1]')
    tn, fp, fn, tp = confusion_matrix(y, p >= threshold, labels=[0, 1]).ravel()
    div = lambda a, b: float(a / b) if b else float('nan')
    sensitivity, specificity = div(tp, tp + fn), div(tn, tn + fp)
    both = len(np.unique(y)) == 2
    bins = np.minimum((p * 10).astype(int), 9)
    ece = sum(np.mean(bins == k) * abs(y[bins == k].mean() - p[bins == k].mean())
              for k in range(10) if (bins == k).any())
    return dict(n=len(y), positives=int(y.sum()), negatives=int((1-y).sum()),
                prevalence=float(y.mean()), threshold=threshold,
                auroc=float(roc_auc_score(y, p)) if both else float('nan'),
                average_precision=float(average_precision_score(y, p)) if both else float('nan'),
                sensitivity=sensitivity, specificity=specificity,
                balanced_accuracy=(sensitivity + specificity) / 2,
                accuracy=div(tp + tn, len(y)), precision=div(tp, tp + fp),
                npv=div(tn, tn + fn), f1=div(2 * tp, 2 * tp + fp + fn),
                brier=float(brier_score_loss(y, p)),
                log_loss=float(log_loss(y, np.clip(p, 1e-7, 1-1e-7), labels=[0, 1])),
                ece_10_bins=float(ece), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))


def attach_metadata(predictions, rows):
    if predictions.image_id.duplicated().any() or set(predictions.image_id) != set(rows.image_id):
        raise ValueError('Prediction identity mismatch or duplication')
    keep = ['image_id', 'binary_label', 'dataset', 'camera', 'patient_id', 'analysis_group', 'classification_image']
    result = predictions.merge(rows[keep], on='image_id', how='left', validate='one_to_one')
    if not np.array_equal(result.label.to_numpy(), result.binary_label.astype(int).to_numpy()):
        raise ValueError('Prediction label alignment failed')
    return result


def grouped_bootstrap(predictions, threshold, seed=2440, iterations=1000):
    """Percentile CIs conditional on a fixed trained model and threshold.

    Resample entire analysis components (patient + duplicate + precautionary ID).
    CHAKSU unknown identities remain unresolved, so its CIs are provisional.
    """
    rng = np.random.default_rng(seed)
    groups = [np.asarray(v) for v in predictions.reset_index(drop=True).groupby('analysis_group').indices.values()]
    if len(groups) < 2:
        raise ValueError('At least two independent groups are needed for a bootstrap')
    y, p = checked_arrays(predictions.label, predictions.probability)
    keys = ['auroc', 'average_precision', 'sensitivity', 'specificity', 'balanced_accuracy', 'brier']
    samples = {key: [] for key in keys}
    for _ in range(iterations):
        indices = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        values = metric_values(y[indices], p[indices], threshold)
        for key in keys:
            if np.isfinite(values[key]):
                samples[key].append(values[key])
    point = metric_values(y, p, threshold)
    records = []
    for key in keys:
        values = samples[key]
        lo, hi = np.quantile(values, [.025, .975]) if len(values) >= 2 else (np.nan, np.nan)
        records.append(dict(metric=key, estimate=point[key], lower_95=lo, upper_95=hi,
                            valid_resamples=len(values), requested_resamples=iterations,
                            bootstrap_groups=len(groups)))
    return pd.DataFrame(records)


def evaluation_tables(predictions, threshold):
    overall = pd.DataFrame([dict(operating_point='default_0.5', **metric_values(predictions.label, predictions.probability, .5)),
                            dict(operating_point='validation_selected', **metric_values(predictions.label, predictions.probability, threshold))])
    cameras = []
    for camera, group in predictions.groupby('camera'):
        values = metric_values(group.label, group.probability, threshold)
        cameras.append(dict(camera=camera, small_class_count=min(values['positives'], values['negatives']) < 10, **values))
    return overall, pd.DataFrame(cameras)


### Plots and single-photo preprocessing

Error galleries retain full fundus images. New photographs receive the same letterboxing as the pipeline.

In [ ]:
def diagnostic_plot(predictions, threshold, title, destination):
    import matplotlib.pyplot as plt
    y, p = checked_arrays(predictions.label, predictions.probability)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fpr, tpr, _ = roc_curve(y, p)
    axes[0].plot(fpr, tpr, label=f'AUROC {roc_auc_score(y, p):.3f}')
    axes[0].plot([0, 1], [0, 1], '--', color='gray')
    axes[0].set(xlabel='False positive rate', ylabel='Sensitivity', title='ROC')
    axes[0].legend()
    precision, recall, _ = precision_recall_curve(y, p)
    axes[1].plot(recall, precision, label=f'AP {average_precision_score(y, p):.3f}')
    axes[1].axhline(y.mean(), linestyle='--', color='gray', label='Prevalence')
    axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision–recall')
    axes[1].legend()
    bins = np.minimum((p * 5).astype(int), 4)
    xs, ys, counts = [], [], []
    for b in range(5):
        mask = bins == b
        if mask.any():
            xs.append(p[mask].mean()); ys.append(y[mask].mean()); counts.append(mask.sum())
    axes[2].plot([0, 1], [0, 1], '--', color='gray')
    axes[2].scatter(xs, ys, s=np.asarray(counts) * 2 + 15)
    axes[2].set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean predicted score', ylabel='Observed positive fraction',
                title='Reliability (marker size = bin count)')
    fig.suptitle(f'{title}; fixed threshold = {threshold:.4f}')
    fig.tight_layout(); fig.savefig(destination, dpi=150, bbox_inches='tight')
    return fig


def error_gallery(predictions, root, threshold, destination, max_each=3):
    import matplotlib.pyplot as plt
    fn = predictions[(predictions.label == 1) & (predictions.probability < threshold)].sort_values('probability').head(max_each)
    fp = predictions[(predictions.label == 0) & (predictions.probability >= threshold)].sort_values('probability', ascending=False).head(max_each)
    examples = pd.concat([fn.assign(error='FN'), fp.assign(error='FP')])
    if examples.empty:
        print('No errors at this operating point.'); return None
    fig, axes = plt.subplots(1, len(examples), figsize=(3.3 * len(examples), 3.5), squeeze=False)
    for ax, row in zip(axes[0], examples.itertuples()):
        with Image.open(image_path(root, row.classification_image)) as im:
            ax.imshow(im.copy())
        ax.set_title(f'{row.error}: y={row.label}, p={row.probability:.3f}\n{row.camera}', fontsize=10)
        ax.set_xlabel(row.image_id, fontsize=6); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout(); fig.savefig(destination, dpi=150, bbox_inches='tight')
    return fig


def letterbox_photo(path):
    # Exactly match prepare_data.padded_resize for new, unprepared images.
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        side = max(im.size)
        canvas = Image.new('RGB', (side, side), 0)
        canvas.paste(im, ((side-im.width)//2, (side-im.height)//2))
        return canvas.resize((224, 224), Image.Resampling.BILINEAR)


### Freezing and final evaluation

Final evaluation reloads the saved checkpoint, saved effective manifest and validation threshold. It writes results once into a separate directory.

In [ ]:
def freeze_run(out, threshold, data_hash, checkpoint):
    out = Path(out)
    path = out / 'frozen_protocol.json'
    record = dict(checkpoint_sha256=sha256_file(out / 'best.pt'),
                  effective_manifest_sha256=data_hash,
                  threshold=float(threshold), threshold_source='CHAKSU validation only',
                  target_validation_sensitivity=checkpoint['config']['target_sensitivity'],
                  model_name=checkpoint['config']['model_name'], weights=checkpoint['weights'],
                  best_epoch=checkpoint['best_epoch'],
                  label_definition='CHAKSU normal vs glaucoma suspect; PAPILA healthy vs clinical glaucoma',
                  probability_note='Uncalibrated sigmoid score; not a validated clinical disease probability')
    if path.exists():
        if json.loads(path.read_text()) != record:
            raise ValueError('Frozen run changed. Use a new development run; never tune on test.')
    else:
        write_json(path, record)
    return record


def verify_frozen(out):
    out = Path(out)
    record = json.loads((out / 'frozen_protocol.json').read_text())
    if sha256_file(out / 'best.pt') != record['checkpoint_sha256']:
        raise ValueError('Checkpoint changed after freezing')
    if sha256_file(out / 'effective_splits/all.csv') != record['effective_manifest_sha256']:
        raise ValueError('Effective manifest changed after freezing')
    return record


def final_evaluation(out, root, device, iterations=1000):
    """Load only the frozen checkpoint, threshold, and saved derivative splits."""
    out = Path(out)
    protocol = verify_frozen(out)
    # Integrity bind per-partition files to the hashed effective manifest.
    all_rows = read_table(out / 'effective_splits/all.csv')
    model, checkpoint = restore_model(out / 'best.pt', device)
    config = checkpoint['config']
    if checkpoint['weights'] == 'NONE_TEST_ONLY':
        warnings.warn('Synthetic software test only: no pretrained weights.')
    final_dir = out / 'final_evaluation'
    final_dir.mkdir(exist_ok=False)
    # If interrupted, inspect this folder; do not silently overwrite test results.
    results = []
    for partition in ('in_domain_test', 'external_test'):
        rows = all_rows[(all_rows.split == partition) & (all_rows.classification_eligible == '1')].copy()
        rows['binary_label'] = rows.binary_label.astype(int)
        check_images(root, rows)
        loader = make_loader(rows, root, config)
        _, pred = run_epoch(model, loader, device)
        pred = attach_metadata(pred, rows)
        pred.to_csv(final_dir / f'{partition}_predictions.csv', index=False)
        overall, cameras = evaluation_tables(pred, protocol['threshold'])
        overall.insert(0, 'partition', partition)
        overall.to_csv(final_dir / f'{partition}_metrics.csv', index=False)
        cameras.to_csv(final_dir / f'{partition}_camera_metrics.csv', index=False)
        grouped_bootstrap(pred, protocol['threshold'], config['seed'], iterations).to_csv(
            final_dir / f'{partition}_confidence_intervals.csv', index=False)
        fig = diagnostic_plot(pred, protocol['threshold'], partition, final_dir / f'{partition}_curves.png')
        import matplotlib.pyplot as plt
        plt.close(fig)
        fig = error_gallery(pred, root, protocol['threshold'], final_dir / f'{partition}_errors.png')
        if fig is not None:
            plt.close(fig)
        results.append(overall)
    result = pd.concat(results, ignore_index=True)
    result.to_csv(final_dir / 'comparison.csv', index=False)
    write_json(final_dir / 'completion.json', dict(protocol=protocol, complete=True))
    return result


## 4. Configure the baseline

Edit **ARTIFACTS_DIR** to the completed pipeline export. The automatic default checks common
locations, including the supplied Downloads project. This is the only path you normally need
to change. `RUNS_DIR` controls where models and plots are saved.

Start with `resnet18`. Change `model_name` to `resnet50` or `efficientnet_b0` only as a separate
development experiment; choose architecture using validation and keep the same effective split.
The explicit ImageNet weight enum is recorded. All architectures use this project's fixed
full-image 224px input and ImageNet normalization; torchvision's default center crop is deliberately
not applied because it would alter the teammate's preprocessing.

Unweighted BCE is the initial training loss. No class weights and no oversampling are used
together or separately in this reference run. Monitor AUROC/AP and use a validation operating
threshold to address screening sensitivity. Weighted BCE may be a later controlled experiment,
but its outputs also require calibration assessment. Do not choose changes based on test results.

In [ ]:
from datetime import datetime, timezone
import uuid
import os
import importlib.metadata

candidates = [Path.cwd() / 'artifacts', Path.cwd().parent / 'artifacts',
              Path.home() / 'Downloads/glaucoma-prediction-data-pipeline/artifacts']
ARTIFACTS_DIR = next((p for p in candidates if (p / 'splits/all.csv').is_file()), candidates[-1])
# ARTIFACTS_DIR = Path('/absolute/path/to/completed/artifacts')  # EDIT if needed
RUNS_DIR = Path.cwd() / 'cnn_baseline_runs'
CONFIG = dict(
    model_name='resnet18', seed=2440, batch_size=16,
    epochs=25, warmup_epochs=2, patience=6,
    head_lr=1e-4, backbone_lr=1e-5, weight_decay=1e-4,
    augment=True, target_sensitivity=0.90,
)
DEVICE_REQUEST = 'auto'  # auto chooses CUDA, then Apple MPS, then CPU
N_BOOTSTRAP = 1000
print('Prepared data:', ARTIFACTS_DIR)
print('Configuration:', CONFIG)


## 5. Audit inputs and save the effective split

This cell stops with an actionable message if the actual data export is missing. It checks
the original manifest hashes, split CSV contents, label meanings, duplicate and patient
separation. It then saves precautionary grouping changes in a **new** run directory. The
source handoff is never overwritten. Do not use old 807/202/336 counts in your report if
the effective counts differ.

The same effective manifests must be reused for future morphology/fusion comparisons.
Rows with invalid reference segmentation are retained for classification; excluding them
would introduce a different, selected classification cohort.

In [ ]:
root, original_rows, pipeline_summary = load_and_audit(ARTIFACTS_DIR)
effective_rows, partitions = effective_partitions(original_rows)

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + CONFIG['model_name'] + '_' + uuid.uuid4().hex[:8]
RUN_DIR = RUNS_DIR / run_id
(RUN_DIR / 'effective_splits').mkdir(parents=True, exist_ok=False)
effective_rows.to_csv(RUN_DIR / 'effective_splits/all.csv', index=False)
for name, frame in partitions.items():
    frame.to_csv(RUN_DIR / 'effective_splits' / f'{name}_classification.csv', index=False)
changes = effective_rows[effective_rows.split != effective_rows.original_split]
changes.to_csv(RUN_DIR / 'effective_splits/changes.csv', index=False)
DATA_HASH = sha256_file(RUN_DIR / 'effective_splits/all.csv')
write_json(RUN_DIR / 'input_audit.json', dict(
    original_summary=pipeline_summary, effective_manifest_sha256=DATA_HASH,
    possible_filename_grouping='same Bosch P-number, irrespective of source release; unverified precaution',
    changed_rows=len(changes), original_root=str(root),
    patient_disjoint_claim=False,
))
write_json(RUN_DIR / 'config.json', CONFIG)
count_rows = []
for name, frame in partitions.items():
    count_rows.append(dict(partition=name, total=len(frame), negatives=int((frame.binary_label == 0).sum()),
                           positives=int((frame.binary_label == 1).sum()), positive_fraction=frame.binary_label.mean()))
counts = pd.DataFrame(count_rows)
counts.to_csv(RUN_DIR / 'effective_splits/counts.csv', index=False)
display(counts)
display(changes[['image_id', 'original_split', 'split', 'split_change_reason']])
print('Original split status:', pipeline_summary.get('status'))
print('Internal results remain provisional: unknown CHAKSU patient linkage is unresolved.')
print('Run directory:', RUN_DIR.resolve())


## 6. Check development images and inspect a training batch

Only train/validation images are opened here. Mild brightness, contrast and saturation
jitter is training-only. No random crops, rotations or flips are applied. Avoid accidentally
loading `overlay_review.png`, `disc_native.png`, segmentation masks or expert measurement files.

In [ ]:
import matplotlib.pyplot as plt
train_rows, val_rows = partitions['train'], partitions['validation']
check_images(root, train_rows)
check_images(root, val_rows)
device = pick_device(DEVICE_REQUEST)
seed_everything(CONFIG['seed'])
versions = {name: importlib.metadata.version(name) for name in
            ['torch', 'torchvision', 'numpy', 'pandas', 'scikit-learn', 'Pillow', 'matplotlib']}
write_json(RUN_DIR / 'environment.json', dict(python=sys.version, packages=versions,
                                            device=str(device), cuda=torch.version.cuda))
print('Device:', device, '\nVersions:', versions)
train_preview = FundusDataset(train_rows, root, training=False)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
# Show both classes without selecting by model results.
sample_indices = train_rows.groupby('binary_label', group_keys=False).sample(n=4, random_state=CONFIG['seed']).index
for ax, i in zip(axes.flat, sample_indices):
    x, y, image_id = train_preview[int(i)]
    rgb = (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).permute(1, 2, 0).numpy()
    ax.imshow(np.clip(rgb, 0, 1)); ax.set_title(f'label={int(y)} | {train_rows.iloc[i].camera}')
    ax.axis('off')
fig.suptitle('Training photographs only; label 1 = CHAKSU glaucoma suspect')
fig.tight_layout(); fig.savefig(RUN_DIR / 'training_examples.png', dpi=150); plt.show()


## 7. Train ResNet-18

The final linear layer outputs one logit per image. `BCEWithLogitsLoss` combines sigmoid
and binary cross-entropy stably, so no sigmoid is placed inside the training model. Sigmoid
is applied when exporting scores. Training uses AdamW, head warm-up, small encoder learning
rate, gradient clipping, and validation early stopping. There is no target-site adaptation.

The best fine-tuning checkpoint is reloaded before evaluation. Training loss is measured on
augmented photographs; validation BCE is measured on deterministic inputs, so these losses
are not a perfectly matched generalization-gap estimate. CPU training works but may be slow.
If you change batch size or any other setting, start a new run and record it.

In [ ]:
model, checkpoint, history = fit_model(
    train_rows, val_rows, root, RUN_DIR, CONFIG, device, pretrained=True)
print('Selected epoch:', checkpoint['best_epoch'])
print('ImageNet initialization:', checkpoint['weights'])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.epoch, history.train_bce, label='Train BCE')
axes[0].plot(history.epoch, history.validation_bce, label='Validation BCE')
axes[0].set(xlabel='Epoch', ylabel='Loss'); axes[0].legend()
axes[1].plot(history.epoch, history.validation_auroc, label='Validation AUROC')
axes[1].plot(history.epoch, history.validation_average_precision, label='Validation AP')
axes[1].set(xlabel='Epoch', ylabel='Ranking metric'); axes[1].legend()
fig.tight_layout(); fig.savefig(RUN_DIR / 'learning_curves.png', dpi=150); plt.show()


## 8. Validation metrics and operating threshold

Save eye-level scores, camera-specific performance, confusion counts, Brier score, log loss,
and 10-bin ECE. Undefined precision or a single-class AUROC is reported as missing rather
than replaced with a misleading zero. Camera results with fewer than 10 examples in either
class are flagged. The original Bosch test had just one positive and Forus had two.

The target of 90% sensitivity is a **validation design choice**, not a clinical guarantee.
Selecting the checkpoint and threshold using the same small validation set makes its reported
performance optimistic. Independent held-out tests are needed. Model outputs are not calibrated
here; reliability plots and Brier score assess calibration rather than assume it.

In [ ]:
_, raw_val_predictions = run_epoch(model, make_loader(val_rows, root, CONFIG), device)
val_predictions = attach_metadata(raw_val_predictions, val_rows)
THRESHOLD = select_threshold(val_predictions.label, val_predictions.probability,
                             CONFIG['target_sensitivity'])
val_predictions['predicted_at_validation_threshold'] = (val_predictions.probability >= THRESHOLD).astype(int)
val_predictions.to_csv(RUN_DIR / 'validation_predictions.csv', index=False)
validation_metrics, validation_cameras = evaluation_tables(val_predictions, THRESHOLD)
validation_metrics.to_csv(RUN_DIR / 'validation_metrics.csv', index=False)
validation_cameras.to_csv(RUN_DIR / 'validation_camera_metrics.csv', index=False)
display(validation_metrics.T)
display(validation_cameras[['camera', 'n', 'positives', 'negatives', 'auroc', 'sensitivity', 'specificity', 'small_class_count']])
diagnostic_plot(val_predictions, THRESHOLD, 'CHAKSU validation (model-selection data)', RUN_DIR / 'validation_curves.png')
plt.show()
error_gallery(val_predictions, root, THRESHOLD, RUN_DIR / 'validation_errors.png')
plt.show()


## 9. Freeze and run final tests only when development is complete

Leave **RUN_FINAL_EVALUATION = False** while changing hyperparameters or comparing
architectures. When the team has selected the final development run, set it to True and
execute this cell. It freezes the checkpoint hash, effective manifest hash and validation
threshold, then evaluates CHAKSU test and PAPILA exactly with those choices.

This procedural switch does not prevent a person from trying other notebooks; keep an
experiment log and do not choose the best model after viewing test performance. Subsequent
method comparisons should be specified in advance. An existing final-results folder is never
silently overwritten. If execution is interrupted, inspect the partial folder before retrying.

Confidence intervals are 95% percentile intervals from 1,000 bootstrap resamples of whole
analysis groups. PAPILA groups preserve patients/bilateral eyes; CHAKSU groups include
known identities, duplicates and precautionary filename links but cannot account for unknown
patient repetition. Intervals condition on this trained model and threshold and do not measure
variation across training seeds. AP, PPV, NPV and calibration also depend on prevalence, so
an external difference must not be attributed solely to camera shift.

In [ ]:
RUN_FINAL_EVALUATION = False  # Set True only for the team's selected final run.
if RUN_FINAL_EVALUATION:
    frozen = freeze_run(RUN_DIR, THRESHOLD, DATA_HASH, checkpoint)
    comparison = final_evaluation(RUN_DIR, root, device, iterations=N_BOOTSTRAP)
    display(comparison[['partition', 'operating_point', 'n', 'positives', 'auroc',
                        'average_precision', 'sensitivity', 'specificity', 'brier']])
    print('Final results:', (RUN_DIR / 'final_evaluation').resolve())
else:
    print('Final tests not run. Only development images have been evaluated.')


## 10. Predict one new full photograph

This cell is optional. Keep it unused on held-out images during development. It applies
the pipeline's full-image letterboxing and normalization and reloads the saved checkpoint.
It returns an uncalibrated suspicion score and the decision at the validation-selected
threshold. No mask or manual disc crop is required.

In [ ]:
NEW_IMAGE = None  # e.g. Path('/path/to/new_fundus.jpg')
if NEW_IMAGE is not None:
    inference_model, _ = restore_model(RUN_DIR / 'best.pt', device)
    prepared = letterbox_photo(NEW_IMAGE)
    x = make_transform(training=False)(prepared).unsqueeze(0).to(device)
    with torch.inference_mode():
        probability = float(torch.sigmoid(inference_model(x).reshape(-1))[0].cpu())
    print({'glaucoma_suspicion_score': probability,
           'flag_at_validation_threshold': probability >= THRESHOLD,
           'threshold': THRESHOLD, 'calibrated_clinical_probability': False})


## 11. How to report this experiment

Describe **pretrained ResNet-18 + full fundus image**, 224px letterboxing, ImageNet
normalization, unweighted BCE, two head-only epochs followed by encoder fine-tuning,
validation-based AUROC selection, and the frozen operating threshold. Report the actual
effective counts and quarantined images from this run. Include the provisional internal
patient split and the CHAKSU/PAPILA label-definition mismatch as limitations.

Copy model numbers from the saved CSVs only. Show one learning-curve figure, ROC/PR curves,
the reliability plot, and representative false negatives/false positives. Inspect what each
failure demonstrates; do not infer early-disease performance without severity annotations.
The existing dataset does not support a blanket claim of clinical deployment readiness.

For future baselines, reuse this effective split. An optic-disc crop can be an independent
ablation only after defining an automatic localizer trained on development data. A crop
derived from a held-out expert mask supplies privileged information and is an oracle analysis.
No crop is included in this full-image reference notebook.

Saved outputs include configuration and package versions, original audit, effective manifests,
changed rows, `history.csv`, `best.pt`, validation predictions/metrics, and diagnostic plots.
The optional final section additionally saves frozen protocol metadata, held-out predictions,
metrics, camera summaries, grouped confidence intervals and plots. Add notebook assistance,
your modifications, and verification steps to the course AI-use log.

### Sources used

Local evidence: `README.md`, `DATA_PREPROCESSING.md`, `config.json`,
`script/prepare_data.py`, `script/load_images.py`, `script/dev/extract_bosch_test_patients.py`,
`script/dev/recompute_duplicates.py`, `patient_groups.csv`, both exploration notebooks,
and the JSON/CSV files in `docs/pipeline_evidence/` from the supplied project.

- [CHAKSU data descriptor](https://doi.org/10.1038/s41597-023-01943-4)
- [PAPILA data descriptor](https://doi.org/10.1038/s41597-022-01388-1)
- [Torchvision ResNet-18 weights and normalization](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [PyTorch transfer-learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [scikit-learn ROC threshold convention](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html)

**Verification boundary:** the delivered source and notebook are software-tested using synthetic
fixtures and cached pretrained weights. Actual CHAKSU/PAPILA training and visual inspection of
their photographs remain pending the missing artifacts export. Synthetic verification is not
a glaucoma performance result.